In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import json, glob, gc
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
def crpic(points, positions, hauteur, largeur):
    return hauteur * (largeur**2) / ((points - positions)**2 + largeur**2)

def normaliser_intensites(df):
    df['Int.'] = df['Int.'] / df['Int.'].max() * 1000
    return df

points_1H  = np.linspace(0, 12,  30000)
points_13C = np.linspace(0, 200, 30000)

In [3]:
SOLVANTS_1H = {'cdcl3': {'ppm': 7.26, 'Int.': 500}, 'ccl4': None,
               'dmso-d6': {'ppm': 2.50, 'Int.': 700}, 'd2o': {'ppm': 4.75, 'Int.': 600}}
SOLVANTS_13C = {'cdcl3': {'ppm': 77.16, 'Int.': 500}, 'ccl4': None,
                'dmso-d6': {'ppm': 39.52, 'Int.': 700}, 'd2o': None}

def charger(dossier_base, solvants):
    d = {}
    for dossier in glob.glob(f'{dossier_base}/*/'):
        ns = Path(dossier).name.lower()
        pic = solvants.get(ns, None)
        for fichier in glob.glob(f'{dossier}*.csv'):
            nom = Path(fichier).stem.split('(')[0]
            try:
                df = pd.read_csv(fichier)[['ppm', 'Int.']]
                df['ppm'] = pd.to_numeric(df['ppm'], errors='coerce')
                df['Int.'] = pd.to_numeric(df['Int.'], errors='coerce')
                df.dropna(inplace=True)
                df = normaliser_intensites(df)
                d[nom] = {'df': df, 'solvant': pic}
            except Exception:
                pass
    return d

molecules_1H  = charger('/content/drive/MyDrive/IA RMN/Molecules CSV 1H',  SOLVANTS_1H)
molecules_13C = charger('/content/drive/MyDrive/IA RMN/Molecules CSV 13C', SOLVANTS_13C)

df_annotations = pd.read_csv('/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/annotations_fonctions.csv',
                              index_col='molecule')
with open('/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/pos_weights.json') as f:
    colonnes_fonctions = json.load(f)['colonnes']
n_fonctions = len(colonnes_fonctions)

freq = df_annotations[colonnes_fonctions].mean().values
pos_weights = np.sqrt((1 - freq) / freq).astype(np.float32)

# Triple intersection : molécules ayant 1H + 13C + annotation
communes = sorted(set(molecules_1H) & set(molecules_13C) & set(df_annotations.index))
print(f"1H: {len(molecules_1H)} | 13C: {len(molecules_13C)} | annotées: {len(df_annotations)}")
print(f"Triple intersection (entraînables) : {len(communes)}")
print(f"Fonctions : {n_fonctions}")

1H: 1385 | 13C: 1244 | annotées: 1231
Triple intersection (entraînables) : 1103
Fonctions : 15


In [4]:
class PaireGenerator(tf.keras.utils.Sequence):
    def __init__(self, communes, mol_1H, mol_13C, df_annotations, colonnes,
                 pts_1H, pts_13C, batch_size=128, n_per_molecule=50, augmentation=True,
                 workers=4, use_multiprocessing=True, max_queue_size=20, **kwargs):
        super().__init__(workers=workers, use_multiprocessing=use_multiprocessing,
                         max_queue_size=max_queue_size, **kwargs)
        self.molecules = communes
        self.pts_1H, self.pts_13C = pts_1H, pts_13C
        self.batch_size = batch_size
        self.augmentation = augmentation
        self.total = len(communes) * n_per_molecule
        self.arr_1H = {m: (mol_1H[m]['df']['ppm'].values, mol_1H[m]['df']['Int.'].values,
                           mol_1H[m]['solvant']) for m in communes}
        self.arr_13C = {m: (mol_13C[m]['df']['ppm'].values, mol_13C[m]['df']['Int.'].values,
                            mol_13C[m]['solvant']) for m in communes}
        self.labels = {m: df_annotations.loc[m, colonnes].values.astype(np.float32)
                       for m in communes}

    def __len__(self):
        return self.total // self.batch_size

    def _gen_1H(self, ppm, ints, solv):
        if self.augmentation:
            xr, yr, lg = np.random.uniform(-0.1,0.1), np.random.uniform(0.8,1.2), np.random.uniform(0.0015,0.004)
        else:
            xr, yr, lg = 0.0, 1.0, 0.002
        s = np.zeros_like(self.pts_1H)
        for p, h in zip(ppm, ints):
            s += crpic(self.pts_1H, p+xr, h*yr, lg)
        if solv is not None and self.augmentation and np.random.random() < 0.7:
            s += crpic(self.pts_1H, solv['ppm']+np.random.uniform(-0.02,0.02),
                       solv['Int.']*np.random.uniform(0.8,1.2), np.random.uniform(0.0015,0.004))
        if self.augmentation:
            a = np.max(s); snr = 10**np.random.uniform(np.log10(150), np.log10(500))
            s += np.random.normal(0, a/snr if a>0 else 0.01, len(self.pts_1H))
        return s

    def _gen_13C(self, ppm, ints, solv):
        if self.augmentation:
            xr, yr, lg = np.random.uniform(-1.0,1.0), np.random.uniform(0.8,1.2), np.random.uniform(0.015,0.025)
        else:
            xr, yr, lg = 0.0, 1.0, 0.02
        s = np.zeros_like(self.pts_13C)
        for p, h in zip(ppm, ints):
            s += crpic(self.pts_13C, p+xr, h*yr, lg)
        if solv is not None and self.augmentation and np.random.random() < 0.7:
            s += crpic(self.pts_13C, solv['ppm']+np.random.uniform(-0.2,0.2),
                       solv['Int.']*np.random.uniform(0.8,1.2), np.random.uniform(0.015,0.025))
        if self.augmentation:
            a = np.max(s); snr = 10**np.random.uniform(np.log10(150), np.log10(500))
            s += np.random.normal(0, a/snr if a>0 else 0.01, len(self.pts_13C))
        return s

    def __getitem__(self, idx):
        X1 = np.empty((self.batch_size, len(self.pts_1H), 1), dtype=np.float32)
        X2 = np.empty((self.batch_size, len(self.pts_13C), 1), dtype=np.float32)
        Y  = np.empty((self.batch_size, n_fonctions), dtype=np.float32)
        for i in range(self.batch_size):
            nom = np.random.choice(self.molecules)
            X1[i,:,0] = self._gen_1H(*self.arr_1H[nom])
            X2[i,:,0] = self._gen_13C(*self.arr_13C[nom])
            Y[i] = self.labels[nom]
        return (X1, X2), Y

In [ ]:
np.random.seed(42)
gen_val_temp = PaireGenerator(communes, molecules_1H, molecules_13C, df_annotations,
                               colonnes_fonctions, points_1H, points_13C,
                               batch_size=128, n_per_molecule=10, augmentation=True,
                               workers=1, use_multiprocessing=False)
X1_list, X2_list, Y_list = [], [], []
for i in range(len(gen_val_temp)):
    (x1, x2), y = gen_val_temp[i]
    X1_list.append(x1); X2_list.append(x2); Y_list.append(y)
X1_val = np.concatenate(X1_list); X2_val = np.concatenate(X2_list); Y_val_ml = np.concatenate(Y_list)
del X1_list, X2_list, Y_list, gen_val_temp; gc.collect()
print(f"Validation : 1H {X1_val.shape}, 13C {X2_val.shape}, Y {Y_val_ml.shape}")

Validation : 1H (11008, 30000, 1), 13C (11008, 30000, 1), Y (11008, 15)


In [ ]:
model_1H  = tf.keras.models.load_model('/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/5_modele_RMN1H.h5')
model_13C = tf.keras.models.load_model('/content/drive/MyDrive/IA RMN/mon_modele_tf 13C/2_modele_RMN13C.h5')
dummy = tf.zeros((1, 30000, 1))
_ = model_1H(dummy); _ = model_13C(dummy)

k_1H  = 11
k_13C = 11

extractor_1H  = tf.keras.Sequential(model_1H.layers[:k_1H+1],  name='embed_1H')
extractor_1H.build((None, 30000, 1));  extractor_1H.trainable = False
extractor_13C = tf.keras.Sequential(model_13C.layers[:k_13C+1], name='embed_13C')
extractor_13C.build((None, 30000, 1)); extractor_13C.trainable = False

print("1H  embedding :", extractor_1H(dummy).shape)   # (1, 64)
print("13C embedding :", extractor_13C(dummy).shape)  # (1, 64)

1H  embedding : (1, 64)
13C embedding : (1, 64)


In [ ]:
in_1H  = tf.keras.Input(shape=(30000, 1), name='spectre_1H')
in_13C = tf.keras.Input(shape=(30000, 1), name='spectre_13C')

e1 = extractor_1H(in_1H, training=False)      # (None, 64)
e2 = extractor_13C(in_13C, training=False)    # (None, 64)

x = tf.keras.layers.Concatenate()([e1, e2])   # (None, 128)
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(32, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(n_fonctions, activation='sigmoid', name='fonctions')(x)

model_fusion = tf.keras.Model(inputs=[in_1H, in_13C], outputs=output)
model_fusion.summary()

Model: "functional_30"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ spectre_1H          │ (None, 30000, 1)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spectre_13C         │ (None, 30000, 1)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed_1H            │ (None, 64)        │    167,712 │ spectre_1H[0][0]  │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed_13C           │ (None, 64)        │    167,712 │ spectre_13C[0][0] │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128)       │          0 │ embed_1H[0][0],   │
│ (Concatenate)       │                   │            │ embed_13C[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      8,256 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,080 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fonctions (Dense)   │ (None, 15)        │        495 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 346,255 (1.32 MB)

 Trainable params: 10,831 (42.31 KB)

 Non-trainable params: 335,424 (1.28 MB)

In [ ]:
def weighted_bce(pos_weights):
    w = tf.constant(pos_weights, dtype=tf.float32)
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        return tf.reduce_mean(-(w*y_true*tf.math.log(y_pred) + (1-y_true)*tf.math.log(1-y_pred)))
    return loss

model_fusion.compile(optimizer='adam', loss=weighted_bce(pos_weights),
                     metrics=[tf.keras.metrics.AUC(name='auc', multi_label=True)])

gen_fusion = PaireGenerator(communes, molecules_1H, molecules_13C, df_annotations,
                             colonnes_fonctions, points_1H, points_13C,
                             batch_size=128, n_per_molecule=50, augmentation=True)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=7, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/modele_fusion.keras',
                                       monitor='val_auc', mode='max', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
]

history = model_fusion.fit(gen_fusion, epochs=30,
                           validation_data=((X1_val, X2_val), Y_val_ml),
                           callbacks=callbacks)

Epoch 1/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 90s 183ms/step - auc: 0.6238 - loss: 0.8408 - val_auc: 0.8267 - val_loss: 0.5572 - learning_rate: 0.0010
Epoch 2/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 71s 164ms/step - auc: 0.8030 - loss: 0.5464 - val_auc: 0.9164 - val_loss: 0.4035 - learning_rate: 0.0010
Epoch 3/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 71s 164ms/step - auc: 0.8740 - loss: 0.4480 - val_auc: 0.9482 - val_loss: 0.3243 - learning_rate: 0.0010
Epoch 4/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 69s 159ms/step - auc: 0.9043 - loss: 0.3936 - val_auc: 0.9626 - val_loss: 0.2768 - learning_rate: 0.0010
Epoch 5/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 70s 160ms/step - auc: 0.9205 - loss: 0.3600 - val_auc: 0.9704 - val_loss: 0.2461 - learning_rate: 0.0010
Epoch 6/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 72s 164ms/step - auc: 0.9312 - loss: 0.3358 - val_auc: 0.9757 - val_loss: 0.2233 - learning_rate: 0.0010
Epoch 7/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 72s 166ms/step - auc: 0.9392 - loss: 0.3156 - val_auc: 0.9788 - val_loss: 0.2077 - learnin

In [ ]:
import os

dossier_sauvegarde = '/content/drive/MyDrive/IA RMN/mon_modele_tf 1H'

if not os.path.exists(dossier_sauvegarde):
    os.makedirs(dossier_sauvegarde)

chemin_h5 = os.path.join(dossier_sauvegarde, 'modele_prediction_fusion.h5')
model_fusion.save(chemin_h5)

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

Y_pred_proba = model_fusion.predict((X1_val, X2_val), batch_size=128, verbose=1)

seuils_optimaux = {}
for i, f in enumerate(colonnes_fonctions):
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.1, 0.91, 0.05):
        s = f1_score(Y_val_ml[:, i], (Y_pred_proba[:, i] >= t).astype(int), zero_division=0)
        if s > best_f1: best_f1, best_t = s, t
    seuils_optimaux[f] = best_t

print(f"{'Fonction':<22} {'Précision':>10} {'Rappel':>8} {'F1':>8} {'seuil':>7}")
print("-" * 60)
Y_opt = np.zeros_like(Y_pred_proba, dtype=int)
for i, f in enumerate(colonnes_fonctions):
    t = seuils_optimaux[f]
    Y_opt[:, i] = (Y_pred_proba[:, i] >= t).astype(int)
    p = precision_score(Y_val_ml[:, i], Y_opt[:, i], zero_division=0)
    r = recall_score(Y_val_ml[:, i], Y_opt[:, i], zero_division=0)
    sc = f1_score(Y_val_ml[:, i], Y_opt[:, i], zero_division=0)
    print(f"  {f:<22} {p:>9.3f} {r:>7.3f} {sc:>7.3f} {t:>6.2f}")

print(f"\nF1 macro FUSION : {f1_score(Y_val_ml, Y_opt, average='macro', zero_division=0):.3f}")

with open('/content/drive/MyDrive/IA RMN/seuils_optimaux_fusion.json', 'w') as f:
    json.dump(seuils_optimaux, f, indent=2)

86/86 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step
Fonction                Précision   Rappel       F1   seuil
------------------------------------------------------------
  aromatique                 0.989   0.986   0.987   0.50
  alcool                     0.931   0.961   0.946   0.55
  phenol                     0.916   0.927   0.921   0.75
  acide_carboxylique         0.915   0.934   0.924   0.55
  cetone                     0.912   0.908   0.910   0.60
  aldehyde                   0.989   0.988   0.989   0.45
  amine                      0.747   0.902   0.817   0.50
  ester                      0.895   0.939   0.917   0.55
  ether                      0.915   0.926   0.921   0.70
  halogenure                 0.853   0.943   0.896   0.65
  nitrile                    0.972   0.953   0.962   0.65
  amide                      0.856   0.878   0.867   0.60
  heterocycle_n              0.930   0.963   0.946   0.55
  sulfoxyde                  0.908   0.942   0.925   0.55
  alcene                   

In [6]:
import json

# Load the fusion model
model_fusion = tf.keras.models.load_model(
    '/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/modele_fusion.keras',
    compile=False
)

# Load the optimal thresholds
with open('/content/drive/MyDrive/IA RMN/seuils_optimaux_fusion.json') as f:
    seuils_fusion = json.load(f)

In [13]:
def predire_fonctions_fusion(csv_1H, csv_13C, n_essais=10):
    # 1H
    df1 = pd.read_csv(csv_1H)[['ppm', 'Int.']]
    df1['ppm']  = pd.to_numeric(df1['ppm'], errors='coerce')
    df1['Int.'] = pd.to_numeric(df1['Int.'], errors='coerce')
    df1.dropna(inplace=True)
    df1 = normaliser_intensites(df1)
    ppm1, int1 = df1['ppm'].values, df1['Int.'].values

    # 13C
    df2 = pd.read_csv(csv_13C)[['ppm', 'Int.']]
    df2['ppm']  = pd.to_numeric(df2['ppm'], errors='coerce')
    df2['Int.'] = pd.to_numeric(df2['Int.'], errors='coerce')
    df2.dropna(inplace=True)
    df2 = normaliser_intensites(df2)
    ppm2, int2 = df2['ppm'].values, df2['Int.'].values

    # Test-time augmentation
    preds = []
    for _ in range(n_essais):
        # 1H Spectrum
        xr, yr, lg = (np.random.uniform(-0.1, 0.1),
                      np.random.uniform(0.8, 1.2),
                      np.random.uniform(0.0015, 0.004))
        s1 = np.zeros_like(points_1H)
        for p, h in zip(ppm1, int1):
            s1 += crpic(points_1H, p + xr, h * yr, lg)
        a1 = np.max(s1)
        snr1 = 10 ** np.random.uniform(np.log10(150), np.log10(500))
        s1 += np.random.normal(0, a1 / snr1 if a1 > 0 else 0.01, len(points_1H))

        # 13C Spectrum
        xr, yr, lg = (np.random.uniform(-1.0, 1.0),
                      np.random.uniform(0.8, 1.2),
                      np.random.uniform(0.015, 0.025))
        s2 = np.zeros_like(points_13C)
        for p, h in zip(ppm2, int2):
            s2 += crpic(points_13C, p + xr, h * yr, lg)
        a2 = np.max(s2)
        snr2 = 10 ** np.random.uniform(np.log10(150), np.log10(500))
        s2 += np.random.normal(0, a2 / snr2 if a2 > 0 else 0.01, len(points_13C))

        x1 = s1[np.newaxis, ..., np.newaxis].astype(np.float32)
        x2 = s2[np.newaxis, ..., np.newaxis].astype(np.float32)
        preds.append(model_fusion((x1, x2), training=False).numpy()[0])

    proba = np.mean(preds, axis=0)

    print(f"Fonctions détectées — FUSION 1H + 13C ({n_essais} tirages) :")
    print("-" * 55)
    detectees = []
    for i, fonction in enumerate(colonnes_fonctions):
        seuil = seuils_fusion[fonction]
        marque = "OUI" if proba[i] >= seuil else "non"
        print(f"  {fonction:<22} {proba[i]*100:5.1f}%  (seuil {seuil:.2f})  {marque}")
        if proba[i] >= seuil:
            detectees.append(fonction)

    print(f"\n→ Fonctions présentes : {', '.join(detectees) if detectees else 'aucune'}")
    return proba


predire_fonctions_fusion(
    '/content/drive/MyDrive/IA RMN/Molecules CSV 1H test/Acetone.csv',
    '/content/drive/MyDrive/IA RMN/Molecules CSV 13C test/Acetone.csv'
)

Fonctions détectées — FUSION 1H + 13C (10 tirages) :
-------------------------------------------------------
  aromatique               0.0%  (seuil 0.50)  non
  alcool                   0.5%  (seuil 0.55)  non
  phenol                   0.0%  (seuil 0.75)  non
  acide_carboxylique       1.0%  (seuil 0.55)  non
  cetone                  97.7%  (seuil 0.60)  OUI
  aldehyde                 0.0%  (seuil 0.45)  non
  amine                   25.6%  (seuil 0.50)  non
  ester                    0.0%  (seuil 0.55)  non
  ether                    0.0%  (seuil 0.70)  non
  halogenure               0.0%  (seuil 0.65)  non
  nitrile                  0.0%  (seuil 0.65)  non
  amide                    0.0%  (seuil 0.60)  non
  heterocycle_n            0.0%  (seuil 0.55)  non
  sulfoxyde                0.0%  (seuil 0.55)  non
  alcene                   0.0%  (seuil 0.65)  non

→ Fonctions présentes : cetone


array([2.5198119e-07, 5.4769856e-03, 1.1102671e-17, 9.8554259e-03,
       9.7748071e-01, 2.0342269e-04, 2.5581110e-01, 2.0122845e-06,
       8.1554735e-10, 3.1660061e-04, 1.9897993e-08, 1.6933607e-05,
       2.4132593e-16, 2.6322940e-07, 4.5992127e-20], dtype=float32)